<a href="https://colab.research.google.com/github/tydormi/fantasy_football/blob/pull-from-colab/Nashville_experiement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: LOAD THE CSV DATA AND SET UP THE PROBLEM ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
budget = 179

# Create the linear programming problem to maximize total points
prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)


# --- STEP 2: DEFINE DECISION VARIABLES AND CONSTRAINTS ---

# A binary variable for each player to decide if they are in the lineup (1) or not (0)
player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

# Objective Function: Maximize the sum of points for selected players
prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

# Constraint 1: The total cost of the lineup cannot exceed the budget
prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= budget, "Budget Constraint"

# Constraint 2: Positional requirements for a valid lineup
# QB: exactly 1
prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB') == 1, "QB Constraint"

# RB: at least 2
prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= 2, "RB Constraint"

# WR: at least 2
prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= 2, "WR Constraint"

# TE: at least 1
prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= 1, "TE Constraint"

# Constraint 3: The flex positions - a total of 7 RB/WR/TE players are required (2 RB, 2 WR, 1 TE, 2 Flex)
flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == 7, "Total RB/WR/TE"

# Constraint 4: The total number of players must be 8
prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == 8, "Total Roster Size"


# --- STEP 3: SOLVE THE PROBLEM AND PRINT THE RESULTS ---

# Solve the optimization problem
prob.solve()

# Check the status of the solution
print("Status:", pulp.LpStatus[prob.status])

# Print the total points and cost of the optimal lineup
total_points = pulp.value(prob.objective)

# Extract the selected players and create a DataFrame
selected_players = [
    eligible_players_df.loc[i]
    for i in eligible_players_df.index
    if player_vars[i].varValue == 1
]
lineup_df = pd.DataFrame(selected_players)

# Calculate the correct total cost by summing the costs of the selected players
total_cost = lineup_df['Cost'].sum()

print(f"Total Projected Points: {total_points}")
print(f"Total Projected Cost: ${total_cost}\n")


# Sort the lineup for a cleaner display
position_order = ['QB', 'RB', 'WR', 'TE']
lineup_df['Position_Order'] = pd.Categorical(
    lineup_df['Position'], categories=position_order, ordered=True
)
lineup_df = lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

print("Optimal Lineup:")
print(lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

Status: Optimal
Total Projected Points: 2031.0
Total Projected Cost: $179

Optimal Lineup:
           Player Position  Points  Cost
   Jayden Daniels       QB     382    28
      Chase Brown       RB     265    36
      Breece Hall       RB     264    35
      Rashee Rice       WR     259    20
    Jaylen Waddle       WR     237    20
Tetairoa McMillan       WR     233    16
      Chris Olave       WR     219    15
   Dalton Kincaid       TE     172     9


In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: DEFINE A FUNCTION TO SOLVE THE OPTIMIZATION PROBLEM ---
def find_optimal_lineup(eligible_players_df, budget):
    """
    Finds the optimal fantasy football lineup using PuLP Linear Programming.

    Args:
        eligible_players_df (pd.DataFrame): DataFrame of available players.
        budget (int): The total budget for the lineup.

    Returns:
        A tuple containing:
        - status (str): The status of the solver ('Optimal', 'Infeasible', etc.).
        - total_points (float): The total projected points of the optimal lineup.
        - total_cost (float): The total cost of the optimal lineup.
        - lineup_df (pd.DataFrame): A DataFrame of the selected players.
    """
    # Create the linear programming problem to maximize total points
    prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)

    # A binary variable for each player to decide if they are in the lineup (1) or not (0)
    player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

    # Objective Function: Maximize the sum of points for selected players
    prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

    # Constraint 1: The total cost of the lineup cannot exceed the budget
    prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= budget, "Budget Constraint"

    # Constraint 2: Positional requirements for a valid lineup
    # QB: exactly 1
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB') == 1, "QB Constraint"

    # RB: at least 2
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= 2, "RB Constraint"

    # WR: at least 2
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= 2, "WR Constraint"

    # TE: at least 1
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= 1, "TE Constraint"

    # Constraint 3: The flex positions - a total of 7 RB/WR/TE players are required (2 RB, 2 WR, 1 TE, 2 Flex)
    flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
    prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == 7, "Total RB/WR/TE"

    # Constraint 4: The total number of players must be 8
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == 8, "Total Roster Size"

    # Solve the optimization problem
    prob.solve()
    status = pulp.LpStatus[prob.status]

    # Extract the selected players and create a DataFrame
    selected_players = [
        eligible_players_df.loc[i]
        for i in eligible_players_df.index
        if player_vars[i].varValue == 1
    ]
    lineup_df = pd.DataFrame(selected_players)

    # Calculate the correct total cost by summing the costs of the selected players
    total_cost = lineup_df['Cost'].sum()
    total_points = pulp.value(prob.objective)

    # Sort the lineup for a cleaner display
    position_order = ['QB', 'RB', 'WR', 'TE']
    lineup_df['Position_Order'] = pd.Categorical(
        lineup_df['Position'], categories=position_order, ordered=True
    )
    lineup_df = lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

    return status, total_points, total_cost, lineup_df


# --- STEP 2: LOAD THE CSV DATA AND INITIALIZE ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
budget = 179

# --- STEP 3: FIND THE INITIAL OPTIMAL LINEUP ---
print("--- Finding Initial Optimal Lineup ---")
status, points, cost, lineup_df = find_optimal_lineup(eligible_players_df, budget)

print(f"Status: {status}")
print(f"Total Projected Points: {points}")
print(f"Total Projected Cost: ${cost}\n")
print("Initial Optimal Lineup:")
print(lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 4: SIMULATE A PLAYER BEING DRAFTED ---
# The logic now becomes dynamic. Let's say 'Jayden Daniels' is drafted by an opponent.
player_drafted = 'Jayden Daniels'
print(f"\n\n--- Simulating draft of '{player_drafted}' by an opponent ---")

# Remove the drafted player from the pool of eligible players
# We'll use a copy to avoid modifying the original DataFrame directly
available_players_df = eligible_players_df[
    eligible_players_df['Player'] != player_drafted
].copy()

# --- STEP 5: RE-OPTIMIZE WITH THE REDUCED PLAYER POOL ---
print("--- Finding New Optimal Lineup ---")
new_status, new_points, new_cost, new_lineup_df = find_optimal_lineup(available_players_df, budget)

print(f"Status: {new_status}")
print(f"New Total Projected Points: {new_points}")
print(f"New Total Projected Cost: ${new_cost}\n")
print(f"New Optimal Lineup (after '{player_drafted}' was drafted):")
print(new_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))


--- Finding Initial Optimal Lineup ---
Status: Optimal
Total Projected Points: 2031.0
Total Projected Cost: $179

Initial Optimal Lineup:
           Player Position  Points  Cost
   Jayden Daniels       QB     382    28
      Chase Brown       RB     265    36
      Breece Hall       RB     264    35
      Rashee Rice       WR     259    20
    Jaylen Waddle       WR     237    20
Tetairoa McMillan       WR     233    16
      Chris Olave       WR     219    15
   Dalton Kincaid       TE     172     9


--- Simulating draft of 'Jayden Daniels' by an opponent ---
--- Finding New Optimal Lineup ---
Status: Optimal
New Total Projected Points: 2029.0
New Total Projected Cost: $179

New Optimal Lineup (after 'Jayden Daniels' was drafted):
           Player Position  Points  Cost
      Jalen Hurts       QB     378    26
      Chase Brown       RB     265    36
      Breece Hall       RB     264    35
      Rashee Rice       WR     259    20
    Jaylen Waddle       WR     237    20
Tetairoa M

In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: DEFINE A FUNCTION TO SOLVE THE OPTIMIZATION PROBLEM ---
def find_optimal_lineup(eligible_players_df, budget, players_to_draft=8):
    """
    Finds the optimal fantasy football lineup using PuLP Linear Programming.

    Args:
        eligible_players_df (pd.DataFrame): DataFrame of available players.
        budget (int): The total budget for the lineup.
        players_to_draft (int): The number of players remaining to draft.

    Returns:
        A tuple containing:
        - status (str): The status of the solver ('Optimal', 'Infeasible', etc.).
        - total_points (float): The total projected points of the optimal lineup.
        - total_cost (float): The total cost of the optimal lineup.
        - lineup_df (pd.DataFrame): A DataFrame of the selected players.
    """
    # Create the linear programming problem to maximize total points
    prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)

    # A binary variable for each player to decide if they are in the lineup (1) or not (0)
    player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

    # Objective Function: Maximize the sum of points for selected players
    prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

    # Constraint 1: The total cost of the lineup cannot exceed the budget
    prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= budget, "Budget Constraint"

    # Constraint 2: Positional requirements for a valid lineup
    # QB: exactly 1
    # This constraint is conditional based on whether a QB has already been drafted
    prob += pulp.lpSum(
        player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB'
    ) == (1 if players_to_draft == 8 else 0), "QB Constraint"

    # RB: at least 2
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= 2, "RB Constraint"

    # WR: at least 2
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= 2, "WR Constraint"

    # TE: at least 1
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= 1, "TE Constraint"

    # Constraint 3: The flex positions - a total of (players_to_draft - 1) RB/WR/TE players are required
    flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
    prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == players_to_draft - (1 if players_to_draft == 8 else 0), "Total RB/WR/TE"

    # Constraint 4: The total number of players must match the number we are drafting
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == players_to_draft, "Total Roster Size"

    # Solve the optimization problem
    prob.solve()
    status = pulp.LpStatus[prob.status]

    # Extract the selected players and create a DataFrame
    selected_players = [
        eligible_players_df.loc[i]
        for i in eligible_players_df.index
        if player_vars[i].varValue == 1
    ]
    lineup_df = pd.DataFrame(selected_players)

    # Calculate the correct total cost by summing the costs of the selected players
    total_cost = lineup_df['Cost'].sum()
    total_points = pulp.value(prob.objective)

    # Sort the lineup for a cleaner display
    position_order = ['QB', 'RB', 'WR', 'TE']
    lineup_df['Position_Order'] = pd.Categorical(
        lineup_df['Position'], categories=position_order, ordered=True
    )
    lineup_df = lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

    return status, total_points, total_cost, lineup_df


# --- STEP 2: LOAD THE CSV DATA AND INITIALIZE ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
initial_budget = 179

# --- STEP 3: FIND THE INITIAL OPTIMAL LINEUP (full 8 players) ---
print("--- Finding Initial Optimal Lineup (Full Roster) ---")
status, points, cost, lineup_df = find_optimal_lineup(eligible_players_df, initial_budget)

print(f"Status: {status}")
print(f"Total Projected Points: {points}")
print(f"Total Projected Cost: ${cost}\n")
print("Initial Optimal Lineup:")
print(lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 4: SIMULATE ME DRAFTING JAYDEN DANIELS FOR $20 ---
player_drafted = 'Jayden Daniels'
actual_cost = 20
new_budget = initial_budget - actual_cost

print(f"\n\n--- Simulating drafting '{player_drafted}' for ${actual_cost} ---")
print(f"New remaining budget: ${new_budget}")

# Remove the drafted player from the pool of eligible players
available_players_df = eligible_players_df[
    eligible_players_df['Player'] != player_drafted
].copy()

# --- STEP 5: RE-OPTIMIZE WITH THE NEW BUDGET AND CONSTRAINTS (7 players) ---
print("--- Finding New Optimal Lineup (Remaining 7 Players) ---")
new_status, new_points, new_cost, new_lineup_df = find_optimal_lineup(available_players_df, new_budget, players_to_draft=7)

print(f"Status: {new_status}")
print(f"New Total Projected Points: {new_points + 382}")
print(f"New Total Projected Cost: ${new_cost + actual_cost}\n")
print(f"New Optimal Lineup (after '{player_drafted}' was drafted):")

# Manually add the drafted player back to the final lineup for display
drafted_player_df = eligible_players_df[eligible_players_df['Player'] == player_drafted].copy()
drafted_player_df.loc[:, 'Cost'] = actual_cost
final_lineup_df = pd.concat([drafted_player_df, new_lineup_df], ignore_index=True)

# Sort the final lineup for a cleaner display
position_order = ['QB', 'RB', 'WR', 'TE']
final_lineup_df['Position_Order'] = pd.Categorical(
    final_lineup_df['Position'], categories=position_order, ordered=True
)
final_lineup_df = final_lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])
print(final_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

--- Finding Initial Optimal Lineup (Full Roster) ---
Status: Optimal
Total Projected Points: 2031.0
Total Projected Cost: $179

Initial Optimal Lineup:
           Player Position  Points  Cost
   Jayden Daniels       QB     382    28
      Chase Brown       RB     265    36
      Breece Hall       RB     264    35
      Rashee Rice       WR     259    20
    Jaylen Waddle       WR     237    20
Tetairoa McMillan       WR     233    16
      Chris Olave       WR     219    15
   Dalton Kincaid       TE     172     9


--- Simulating drafting 'Jayden Daniels' for $20 ---
New remaining budget: $159
--- Finding New Optimal Lineup (Remaining 7 Players) ---
Status: Optimal
New Total Projected Points: 2048.0
New Total Projected Cost: $178

New Optimal Lineup (after 'Jayden Daniels' was drafted):
           Player Position  Points  Cost
   Jayden Daniels       QB     382    20
      Chase Brown       RB     265    36
      Breece Hall       RB     264    35
      Rashee Rice       WR     259  

In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: DEFINE A FUNCTION TO SOLVE THE OPTIMIZATION PROBLEM ---
def find_optimal_lineup(eligible_players_df, drafted_players, initial_budget):
    """
    Finds the optimal fantasy football lineup using PuLP Linear Programming.

    This function dynamically adjusts constraints based on the players already drafted.

    Args:
        eligible_players_df (pd.DataFrame): DataFrame of available players.
        drafted_players (pd.DataFrame): DataFrame of players already on your team.
        initial_budget (int): The starting budget for the entire draft.

    Returns:
        A tuple containing:
        - status (str): The status of the solver ('Optimal', 'Infeasible', etc.).
        - total_projected_points (float): The total projected points of the final lineup.
        - total_actual_cost (float): The total cost of the final lineup.
        - final_lineup_df (pd.DataFrame): The complete DataFrame of the selected players.
    """
    # Calculate current state based on drafted players
    players_to_draft = 8 - len(drafted_players)
    budget_spent = drafted_players['Cost'].sum()
    remaining_budget = initial_budget - budget_spent

    # Calculate remaining positional needs
    qbs_needed = 1 - len(drafted_players[drafted_players['Position'] == 'QB'])
    rbs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'RB'])
    wrs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'WR'])
    tes_needed = 1 - len(drafted_players[drafted_players['Position'] == 'TE'])

    # Ensure needs are not negative
    qbs_needed = max(0, qbs_needed)
    rbs_needed = max(0, rbs_needed)
    wrs_needed = max(0, wrs_needed)
    tes_needed = max(0, tes_needed)

    # Calculate total RB/WR/TE spots needed for the new problem
    flex_players_drafted = len(drafted_players[drafted_players['Position'].isin(['RB', 'WR', 'TE'])])
    flex_players_needed = 7 - flex_players_drafted

    # Create the linear programming problem to maximize total points
    prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)

    # A binary variable for each player to decide if they are in the lineup (1) or not (0)
    player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

    # Objective Function: Maximize the sum of points for selected players
    prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

    # Constraint 1: The total cost of the new players cannot exceed the remaining budget
    prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= remaining_budget, "Remaining Budget Constraint"

    # Constraint 2: Positional requirements for a valid lineup
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB') == qbs_needed, "QB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= rbs_needed, "RB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= wrs_needed, "WR Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= tes_needed, "TE Constraint"

    # Constraint 3: The flex positions - a total of (players_to_draft - qbs_needed) RB/WR/TE players are required
    flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
    prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == flex_players_needed, "Total RB/WR/TE"

    # Constraint 4: The total number of players must match the number we are drafting
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == players_to_draft, "Total Roster Size"

    # Solve the optimization problem
    prob.solve()
    status = pulp.LpStatus[prob.status]

    # Extract the newly selected players and create a DataFrame
    newly_selected_players = [
        eligible_players_df.loc[i]
        for i in eligible_players_df.index
        if player_vars[i].varValue == 1
    ]
    newly_selected_df = pd.DataFrame(newly_selected_players)

    # Combine the drafted players with the newly selected optimal players
    final_lineup_df = pd.concat([drafted_players, newly_selected_df], ignore_index=True)

    # Calculate the total points and cost of the final lineup
    total_projected_points = final_lineup_df['Points'].sum()
    total_actual_cost = final_lineup_df['Cost'].sum()

    # Sort the final lineup for a cleaner display
    position_order = ['QB', 'RB', 'WR', 'TE']
    final_lineup_df['Position_Order'] = pd.Categorical(
        final_lineup_df['Position'], categories=position_order, ordered=True
    )
    final_lineup_df = final_lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

    return status, total_projected_points, total_actual_cost, final_lineup_df


# --- STEP 2: LOAD THE CSV DATA AND INITIALIZE ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
initial_budget = 179

# --- STEP 3: FIND THE INITIAL OPTIMAL LINEUP (full 8 players) ---
print("--- Finding Initial Optimal Lineup (Full Roster) ---")
# Start with an empty drafted players DataFrame with the correct columns
drafted_players_df = pd.DataFrame(columns=['Player', 'Position', 'Points', 'Cost'])
status, points, cost, final_lineup_df = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {status}")
print(f"Total Projected Points: {points}")
print(f"Total Projected Cost: ${cost}\n")
print("Initial Optimal Lineup:")
print(final_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 4: SIMULATE MY DRAFTING JAYDEN DANIELS FOR $20 ---
player_drafted = 'Jayden Daniels'
actual_cost = 20

print(f"\n\n--- Simulating drafting '{player_drafted}' for ${actual_cost} ---")

# Add the drafted player to the team with the actual cost
drafted_player_series = eligible_players_df[eligible_players_df['Player'] == player_drafted].iloc[0].copy()
drafted_player_series['Cost'] = actual_cost
drafted_players_df = pd.concat([drafted_players_df, drafted_player_series.to_frame().T], ignore_index=True)

# Remove the drafted player from the pool of eligible players
eligible_players_df = eligible_players_df[
    eligible_players_df['Player'] != player_drafted
].copy().reset_index(drop=True)

# --- STEP 5: RE-OPTIMIZE WITH THE NEW BUDGET AND CONSTRAINTS ---
print("--- Finding New Optimal Lineup (Remaining 7 Players) ---")
new_status, new_points, new_cost, new_lineup_df = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {new_status}")
print(f"New Total Projected Points: {new_points}")
print(f"New Total Projected Cost: ${new_cost}\n")
print(f"New Optimal Lineup (after '{player_drafted}' was drafted):")
print(new_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))


--- Finding Initial Optimal Lineup (Full Roster) ---
Status: Optimal
Total Projected Points: 2031
Total Projected Cost: $179

Initial Optimal Lineup:
           Player Position Points Cost
   Jayden Daniels       QB    382   28
      Chase Brown       RB    265   36
      Breece Hall       RB    264   35
      Rashee Rice       WR    259   20
    Jaylen Waddle       WR    237   20
Tetairoa McMillan       WR    233   16
      Chris Olave       WR    219   15
   Dalton Kincaid       TE    172    9


--- Simulating drafting 'Jayden Daniels' for $20 ---
--- Finding New Optimal Lineup (Remaining 7 Players) ---
Status: Optimal
New Total Projected Points: 2048
New Total Projected Cost: $178

New Optimal Lineup (after 'Jayden Daniels' was drafted):
           Player Position Points Cost
   Jayden Daniels       QB    382   20
      Chase Brown       RB    265   36
      Breece Hall       RB    264   35
      Rashee Rice       WR    259   20
    Jaylen Waddle       WR    237   20
    DeVonta Smi

In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: DEFINE A FUNCTION TO SOLVE THE OPTIMIZATION PROBLEM ---
def find_optimal_lineup(eligible_players_df, drafted_players, initial_budget):
    """
    Finds the optimal fantasy football lineup using PuLP Linear Programming.

    This function dynamically adjusts constraints based on the players already drafted.

    Args:
        eligible_players_df (pd.DataFrame): DataFrame of available players.
        drafted_players (pd.DataFrame): DataFrame of players already on your team.
        initial_budget (int): The starting budget for the entire draft.

    Returns:
        A tuple containing:
        - status (str): The status of the solver ('Optimal', 'Infeasible', etc.).
        - total_projected_points (float): The total projected points of the final lineup.
        - total_actual_cost (float): The total cost of the final lineup.
        - final_lineup_df (pd.DataFrame): The complete DataFrame of the selected players.
    """
    # Calculate current state based on drafted players
    players_to_draft = 8 - len(drafted_players)
    budget_spent = drafted_players['Cost'].sum()
    remaining_budget = initial_budget - budget_spent

    # Calculate remaining positional needs
    qbs_needed = 1 - len(drafted_players[drafted_players['Position'] == 'QB'])
    rbs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'RB'])
    wrs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'WR'])
    tes_needed = 1 - len(drafted_players[drafted_players['Position'] == 'TE'])

    # Ensure needs are not negative
    qbs_needed = max(0, qbs_needed)
    rbs_needed = max(0, rbs_needed)
    wrs_needed = max(0, wrs_needed)
    tes_needed = max(0, tes_needed)

    # Calculate total RB/WR/TE spots needed for the new problem
    flex_players_drafted = len(drafted_players[drafted_players['Position'].isin(['RB', 'WR', 'TE'])])
    flex_players_needed = 7 - flex_players_drafted

    # Create the linear programming problem to maximize total points
    prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)

    # A binary variable for each player to decide if they are in the lineup (1) or not (0)
    player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

    # Objective Function: Maximize the sum of points for selected players
    prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

    # Constraint 1: The total cost of the new players cannot exceed the remaining budget
    prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= remaining_budget, "Remaining Budget Constraint"

    # Constraint 2: Positional requirements for a valid lineup
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB') == qbs_needed, "QB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= rbs_needed, "RB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= wrs_needed, "WR Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= tes_needed, "TE Constraint"

    # Constraint 3: The flex positions - a total of (players_to_draft - qbs_needed) RB/WR/TE players are required
    flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
    prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == flex_players_needed, "Total RB/WR/TE"

    # Constraint 4: The total number of players must match the number we are drafting
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == players_to_draft, "Total Roster Size"

    # Solve the optimization problem
    prob.solve()
    status = pulp.LpStatus[prob.status]

    # Extract the newly selected players and create a DataFrame
    newly_selected_players = [
        eligible_players_df.loc[i]
        for i in eligible_players_df.index
        if player_vars[i].varValue == 1
    ]
    newly_selected_df = pd.DataFrame(newly_selected_players)

    # Combine the drafted players with the newly selected optimal players
    final_lineup_df = pd.concat([drafted_players, newly_selected_df], ignore_index=True)

    # Calculate the total points and cost of the final lineup
    total_projected_points = final_lineup_df['Points'].sum()
    total_actual_cost = final_lineup_df['Cost'].sum()

    # Sort the final lineup for a cleaner display
    position_order = ['QB', 'RB', 'WR', 'TE']
    final_lineup_df['Position_Order'] = pd.Categorical(
        final_lineup_df['Position'], categories=position_order, ordered=True
    )
    final_lineup_df = final_lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

    return status, total_projected_points, total_actual_cost, final_lineup_df


# --- STEP 2: LOAD THE CSV DATA AND INITIALIZE ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
initial_budget = 179

# Start with an empty drafted players DataFrame with the correct columns
drafted_players_df = pd.DataFrame(columns=['Player', 'Position', 'Points', 'Cost'])

# --- STEP 3: FIND THE INITIAL OPTIMAL LINEUP (full 8 players) ---
print("--- Finding Initial Optimal Lineup (Full Roster) ---")
status, points, cost, final_lineup_df = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {status}")
print(f"Total Projected Points: {points}")
print(f"Total Projected Cost: ${cost}\n")
print("Initial Optimal Lineup:")
print(final_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 4: SIMULATE DRAFTING JAYDEN DANIELS FOR $20 ---
player_drafted_1 = 'Jayden Daniels'
actual_cost_1 = 20

print(f"\n\n--- Simulating drafting '{player_drafted_1}' for ${actual_cost_1} ---")

# Add the first drafted player to the team with the actual cost
drafted_player_series_1 = eligible_players_df[eligible_players_df['Player'] == player_drafted_1].iloc[0].copy()
drafted_player_series_1['Cost'] = actual_cost_1
drafted_players_df = pd.concat([drafted_players_df, drafted_player_series_1.to_frame().T], ignore_index=True)

# Remove the drafted player from the pool of eligible players
eligible_players_df = eligible_players_df[
    eligible_players_df['Player'] != player_drafted_1
].copy().reset_index(drop=True)

# --- STEP 5: RE-OPTIMIZE WITH THE NEW BUDGET AND CONSTRAINTS (7 players) ---
print("--- Finding New Optimal Lineup (Remaining 7 Players) ---")
new_status_1, new_points_1, new_cost_1, new_lineup_df_1 = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {new_status_1}")
print(f"New Total Projected Points: {new_points_1}")
print(f"New Total Projected Cost: ${new_cost_1}\n")
print(f"New Optimal Lineup (after '{player_drafted_1}' was drafted):")
print(new_lineup_df_1[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 6: SIMULATE DRAFTING CHASE BROWN FOR $38 ---
player_drafted_2 = 'Chase Brown'
actual_cost_2 = 38

print(f"\n\n--- Simulating drafting '{player_drafted_2}' for ${actual_cost_2} ---")

# Add the second drafted player to the team with the actual cost
drafted_player_series_2 = eligible_players_df[eligible_players_df['Player'] == player_drafted_2].iloc[0].copy()
drafted_player_series_2['Cost'] = actual_cost_2
drafted_players_df = pd.concat([drafted_players_df, drafted_player_series_2.to_frame().T], ignore_index=True)

# Remove the drafted player from the pool of eligible players
eligible_players_df = eligible_players_df[
    eligible_players_df['Player'] != player_drafted_2
].copy().reset_index(drop=True)

# --- STEP 7: RE-OPTIMIZE WITH THE NEW BUDGET AND CONSTRAINTS (6 players) ---
print("--- Finding New Optimal Lineup (Remaining 6 Players) ---")
new_status_2, new_points_2, new_cost_2, new_lineup_df_2 = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {new_status_2}")
print(f"New Total Projected Points: {new_points_2}")
print(f"New Total Projected Cost: ${new_cost_2}\n")
print(f"New Optimal Lineup (after '{player_drafted_1}' and '{player_drafted_2}' were drafted):")
print(new_lineup_df_2[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))


--- Finding Initial Optimal Lineup (Full Roster) ---
Status: Optimal
Total Projected Points: 2031
Total Projected Cost: $179

Initial Optimal Lineup:
           Player Position Points Cost
   Jayden Daniels       QB    382   28
      Chase Brown       RB    265   36
      Breece Hall       RB    264   35
      Rashee Rice       WR    259   20
    Jaylen Waddle       WR    237   20
Tetairoa McMillan       WR    233   16
      Chris Olave       WR    219   15
   Dalton Kincaid       TE    172    9


--- Simulating drafting 'Jayden Daniels' for $20 ---
--- Finding New Optimal Lineup (Remaining 7 Players) ---
Status: Optimal
New Total Projected Points: 2048
New Total Projected Cost: $178

New Optimal Lineup (after 'Jayden Daniels' was drafted):
           Player Position Points Cost
   Jayden Daniels       QB    382   20
      Chase Brown       RB    265   36
      Breece Hall       RB    264   35
      Rashee Rice       WR    259   20
    Jaylen Waddle       WR    237   20
    DeVonta Smi

In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: DEFINE A FUNCTION TO SOLVE THE OPTIMIZATION PROBLEM ---
def find_optimal_lineup(eligible_players_df, drafted_players, initial_budget):
    """
    Finds the optimal fantasy football lineup using PuLP Linear Programming.

    This function dynamically adjusts constraints based on the players already drafted.

    Args:
        eligible_players_df (pd.DataFrame): DataFrame of available players.
        drafted_players (pd.DataFrame): DataFrame of players already on your team.
        initial_budget (int): The starting budget for the entire draft.

    Returns:
        A tuple containing:
        - status (str): The status of the solver ('Optimal', 'Infeasible', etc.).
        - total_projected_points (float): The total projected points of the final lineup.
        - total_actual_cost (float): The total cost of the final lineup.
        - final_lineup_df (pd.DataFrame): The complete DataFrame of the selected players.
    """
    # Calculate current state based on drafted players
    players_to_draft = 8 - len(drafted_players)
    budget_spent = drafted_players['Cost'].sum()
    remaining_budget = initial_budget - budget_spent

    # Calculate remaining positional needs
    qbs_needed = 1 - len(drafted_players[drafted_players['Position'] == 'QB'])
    rbs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'RB'])
    wrs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'WR'])
    tes_needed = 1 - len(drafted_players[drafted_players['Position'] == 'TE'])

    # Ensure needs are not negative
    qbs_needed = max(0, qbs_needed)
    rbs_needed = max(0, rbs_needed)
    wrs_needed = max(0, wrs_needed)
    tes_needed = max(0, tes_needed)

    # Calculate total RB/WR/TE spots needed for the new problem
    flex_players_drafted = len(drafted_players[drafted_players['Position'].isin(['RB', 'WR', 'TE'])])
    flex_players_needed = 7 - flex_players_drafted

    # Create the linear programming problem to maximize total points
    prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)

    # A binary variable for each player to decide if they are in the lineup (1) or not (0)
    player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

    # Objective Function: Maximize the sum of points for selected players
    prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

    # Constraint 1: The total cost of the new players cannot exceed the remaining budget
    prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= remaining_budget, "Remaining Budget Constraint"

    # Constraint 2: Positional requirements for a valid lineup
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB') == qbs_needed, "QB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= rbs_needed, "RB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= wrs_needed, "WR Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= tes_needed, "TE Constraint"

    # Constraint 3: The flex positions - a total of (players_to_draft - qbs_needed) RB/WR/TE players are required
    flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
    prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == flex_players_needed, "Total RB/WR/TE"

    # Constraint 4: The total number of players must match the number we are drafting
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == players_to_draft, "Total Roster Size"

    # Solve the optimization problem
    prob.solve()
    status = pulp.LpStatus[prob.status]

    # Extract the newly selected players and create a DataFrame
    newly_selected_players = [
        eligible_players_df.loc[i]
        for i in eligible_players_df.index
        if player_vars[i].varValue == 1
    ]
    newly_selected_df = pd.DataFrame(newly_selected_players)

    # Combine the drafted players with the newly selected optimal players
    final_lineup_df = pd.concat([drafted_players, newly_selected_df], ignore_index=True)

    # Calculate the total points and cost of the final lineup
    total_projected_points = final_lineup_df['Points'].sum()
    total_actual_cost = final_lineup_df['Cost'].sum()

    # Sort the final lineup for a cleaner display
    position_order = ['QB', 'RB', 'WR', 'TE']
    final_lineup_df['Position_Order'] = pd.Categorical(
        final_lineup_df['Position'], categories=position_order, ordered=True
    )
    final_lineup_df = final_lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

    return status, total_projected_points, total_actual_cost, final_lineup_df


# --- STEP 2: LOAD THE CSV DATA AND INITIALIZE ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
initial_budget = 179

# Start with an empty drafted players DataFrame with the correct columns
drafted_players_df = pd.DataFrame(columns=['Player', 'Position', 'Points', 'Cost'])

# --- STEP 3: FIND THE INITIAL OPTIMAL LINEUP (full 8 players) ---
print("--- Finding Initial Optimal Lineup (Full Roster) ---")
status, points, cost, final_lineup_df = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {status}")
print(f"Total Projected Points: {points}")
print(f"Total Projected Cost: ${cost}\n")
print("Initial Optimal Lineup:")
print(final_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 4: SIMULATE DRAFTING JAYDEN DANIELS FOR $20 ---
player_drafted_1 = 'Jayden Daniels'
actual_cost_1 = 20

print(f"\n\n--- Simulating drafting '{player_drafted_1}' for ${actual_cost_1} ---")

# Add the first drafted player to the team with the actual cost
drafted_player_series_1 = eligible_players_df[eligible_players_df['Player'] == player_drafted_1].iloc[0].copy()
drafted_player_series_1['Cost'] = actual_cost_1
drafted_players_df = pd.concat([drafted_players_df, drafted_player_series_1.to_frame().T], ignore_index=True)

# Remove the drafted player from the pool of eligible players
eligible_players_df = eligible_players_df[
    eligible_players_df['Player'] != player_drafted_1
].copy().reset_index(drop=True)

# --- STEP 5: RE-OPTIMIZE WITH THE NEW BUDGET AND CONSTRAINTS (7 players) ---
print("--- Finding New Optimal Lineup (Remaining 7 Players) ---")
new_status_1, new_points_1, new_cost_1, new_lineup_df_1 = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {new_status_1}")
print(f"New Total Projected Points: {new_points_1}")
print(f"New Total Projected Cost: ${new_cost_1}\n")
print(f"New Optimal Lineup (after '{player_drafted_1}' was drafted):")
print(new_lineup_df_1[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 6: SIMULATE DRAFTING CHASE BROWN FOR $40 ---
player_drafted_2 = 'Chase Brown'
actual_cost_2 = 40

print(f"\n\n--- Simulating drafting '{player_drafted_2}' for ${actual_cost_2} ---")

# Add the second drafted player to the team with the actual cost
drafted_player_series_2 = eligible_players_df[eligible_players_df['Player'] == player_drafted_2].iloc[0].copy()
drafted_player_series_2['Cost'] = actual_cost_2
drafted_players_df = pd.concat([drafted_players_df, drafted_player_series_2.to_frame().T], ignore_index=True)

# Remove the drafted player from the pool of eligible players
eligible_players_df = eligible_players_df[
    eligible_players_df['Player'] != player_drafted_2
].copy().reset_index(drop=True)

# --- STEP 7: RE-OPTIMIZE WITH THE NEW BUDGET AND CONSTRAINTS (6 players) ---
print("--- Finding New Optimal Lineup (Remaining 6 Players) ---")
new_status_2, new_points_2, new_cost_2, new_lineup_df_2 = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {new_status_2}")
print(f"New Total Projected Points: {new_points_2}")
print(f"New Total Projected Cost: ${new_cost_2}\n")
print(f"New Optimal Lineup (after '{player_drafted_1}' and '{player_drafted_2}' were drafted):")
print(new_lineup_df_2[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

--- Finding Initial Optimal Lineup (Full Roster) ---
Status: Optimal
Total Projected Points: 2031
Total Projected Cost: $179

Initial Optimal Lineup:
           Player Position Points Cost
   Jayden Daniels       QB    382   28
      Chase Brown       RB    265   36
      Breece Hall       RB    264   35
      Rashee Rice       WR    259   20
    Jaylen Waddle       WR    237   20
Tetairoa McMillan       WR    233   16
      Chris Olave       WR    219   15
   Dalton Kincaid       TE    172    9


--- Simulating drafting 'Jayden Daniels' for $20 ---
--- Finding New Optimal Lineup (Remaining 7 Players) ---
Status: Optimal
New Total Projected Points: 2048
New Total Projected Cost: $178

New Optimal Lineup (after 'Jayden Daniels' was drafted):
           Player Position Points Cost
   Jayden Daniels       QB    382   20
      Chase Brown       RB    265   36
      Breece Hall       RB    264   35
      Rashee Rice       WR    259   20
    Jaylen Waddle       WR    237   20
    DeVonta Smi

In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: DEFINE A FUNCTION TO SOLVE THE OPTIMIZATION PROBLEM ---
def find_optimal_lineup(eligible_players_df, drafted_players, initial_budget):
    """
    Finds the optimal fantasy football lineup using PuLP Linear Programming.

    This function dynamically adjusts constraints based on the players already drafted.

    Args:
        eligible_players_df (pd.DataFrame): DataFrame of available players.
        drafted_players (pd.DataFrame): DataFrame of players already on your team.
        initial_budget (int): The starting budget for the entire draft.

    Returns:
        A tuple containing:
        - status (str): The status of the solver ('Optimal', 'Infeasible', etc.).
        - total_projected_points (float): The total projected points of the final lineup.
        - total_actual_cost (float): The total cost of the final lineup.
        - final_lineup_df (pd.DataFrame): The complete DataFrame of the selected players.
    """
    # Calculate current state based on drafted players
    players_to_draft = 8 - len(drafted_players)
    budget_spent = drafted_players['Cost'].sum()
    remaining_budget = initial_budget - budget_spent

    # Calculate remaining positional needs
    qbs_needed = 1 - len(drafted_players[drafted_players['Position'] == 'QB'])
    rbs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'RB'])
    wrs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'WR'])
    tes_needed = 1 - len(drafted_players[drafted_players['Position'] == 'TE'])

    # Ensure needs are not negative
    qbs_needed = max(0, qbs_needed)
    rbs_needed = max(0, rbs_needed)
    wrs_needed = max(0, wrs_needed)
    tes_needed = max(0, tes_needed)

    # Calculate total RB/WR/TE spots needed for the new problem
    flex_players_drafted = len(drafted_players[drafted_players['Position'].isin(['RB', 'WR', 'TE'])])
    flex_players_needed = 7 - flex_players_drafted

    # Create the linear programming problem to maximize total points
    prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)

    # A binary variable for each player to decide if they are in the lineup (1) or not (0)
    player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

    # Objective Function: Maximize the sum of points for selected players
    prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

    # Constraint 1: The total cost of the new players cannot exceed the remaining budget
    prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= remaining_budget, "Remaining Budget Constraint"

    # Constraint 2: Positional requirements for a valid lineup
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB') == qbs_needed, "QB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= rbs_needed, "RB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= wrs_needed, "WR Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= tes_needed, "TE Constraint"

    # Constraint 3: The flex positions - a total of (players_to_draft - qbs_needed) RB/WR/TE players are required
    flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
    prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == flex_players_needed, "Total RB/WR/TE"

    # Constraint 4: The total number of players must match the number we are drafting
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == players_to_draft, "Total Roster Size"

    # Solve the optimization problem
    prob.solve()
    status = pulp.LpStatus[prob.status]

    # Extract the newly selected players and create a DataFrame
    newly_selected_players = [
        eligible_players_df.loc[i]
        for i in eligible_players_df.index
        if player_vars[i].varValue == 1
    ]
    newly_selected_df = pd.DataFrame(newly_selected_players)

    # Combine the drafted players with the newly selected optimal players
    final_lineup_df = pd.concat([drafted_players, newly_selected_df], ignore_index=True)

    # Calculate the total points and cost of the final lineup
    total_projected_points = final_lineup_df['Points'].sum()
    total_actual_cost = final_lineup_df['Cost'].sum()

    # Sort the final lineup for a cleaner display
    position_order = ['QB', 'RB', 'WR', 'TE']
    final_lineup_df['Position_Order'] = pd.Categorical(
        final_lineup_df['Position'], categories=position_order, ordered=True
    )
    final_lineup_df = final_lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

    return status, total_projected_points, total_actual_cost, final_lineup_df


# --- STEP 2: LOAD THE CSV DATA AND INITIALIZE ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
initial_budget = 179

# Start with an empty drafted players DataFrame with the correct columns
drafted_players_df = pd.DataFrame(columns=['Player', 'Position', 'Points', 'Cost'])

# --- STEP 3: FIND THE INITIAL OPTIMAL LINEUP (full 8 players) ---
print("--- Finding Initial Optimal Lineup (Full Roster) ---")
status, points, cost, final_lineup_df = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {status}")
print(f"Total Projected Points: {points}")
print(f"Total Projected Cost: ${cost}\n")
print("Initial Optimal Lineup:")
print(final_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 4: SIMULATE DRAFTING BUCKY IRVING FOR $5 ---
player_drafted_1 = 'Bucky Irving'
actual_cost_1 = 5

print(f"\n\n--- Simulating drafting '{player_drafted_1}' for ${actual_cost_1} ---")

# Add the first drafted player to the team with the actual cost
drafted_player_series_1 = eligible_players_df[eligible_players_df['Player'] == player_drafted_1].iloc[0].copy()
drafted_player_series_1['Cost'] = actual_cost_1
drafted_players_df = pd.concat([drafted_players_df, drafted_player_series_1.to_frame().T], ignore_index=True)

# Remove the drafted player from the pool of eligible players
eligible_players_df = eligible_players_df[
    eligible_players_df['Player'] != player_drafted_1
].copy().reset_index(drop=True)

# --- STEP 5: SIMULATE DRAFTING PUKA NACUA FOR $10 ---
player_drafted_2 = 'Puka Nacua'
actual_cost_2 = 10

print(f"\n\n--- Simulating drafting '{player_drafted_2}' for ${actual_cost_2} ---")

# Add the second drafted player to the team with the actual cost
drafted_player_series_2 = eligible_players_df[eligible_players_df['Player'] == player_drafted_2].iloc[0].copy()
drafted_player_series_2['Cost'] = actual_cost_2
drafted_players_df = pd.concat([drafted_players_df, drafted_player_series_2.to_frame().T], ignore_index=True)

# Remove the drafted player from the pool of eligible players
eligible_players_df = eligible_players_df[
    eligible_players_df['Player'] != player_drafted_2
].copy().reset_index(drop=True)

# --- STEP 6: SIMULATE DRAFTING JA'MARR CHASE FOR $64 ---
player_drafted_3 = 'Ja\'Marr Chase'
actual_cost_3 = 64

print(f"\n\n--- Simulating drafting '{player_drafted_3}' for ${actual_cost_3} ---")

# Add the third drafted player to the team with the actual cost
drafted_player_series_3 = eligible_players_df[eligible_players_df['Player'] == player_drafted_3].iloc[0].copy()
drafted_player_series_3['Cost'] = actual_cost_3
drafted_players_df = pd.concat([drafted_players_df, drafted_player_series_3.to_frame().T], ignore_index=True)

# Remove the drafted player from the pool of eligible players
eligible_players_df = eligible_players_df[
    eligible_players_df['Player'] != player_drafted_3
].copy().reset_index(drop=True)

# --- STEP 7: RE-OPTIMIZE WITH THE NEW BUDGET AND CONSTRAINTS (5 players) ---
print("--- Finding New Optimal Lineup (Remaining 5 Players) ---")
new_status, new_points, new_cost, new_lineup_df = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {new_status}")
print(f"New Total Projected Points: {new_points}")
print(f"New Total Projected Cost: ${new_cost}\n")
print(f"New Optimal Lineup (after '{player_drafted_1}', '{player_drafted_2}', and '{player_drafted_3}' were drafted):")
print(new_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))


--- Finding Initial Optimal Lineup (Full Roster) ---
Status: Optimal
Total Projected Points: 2031
Total Projected Cost: $179

Initial Optimal Lineup:
           Player Position Points Cost
   Jayden Daniels       QB    382   28
      Chase Brown       RB    265   36
      Breece Hall       RB    264   35
      Rashee Rice       WR    259   20
    Jaylen Waddle       WR    237   20
Tetairoa McMillan       WR    233   16
      Chris Olave       WR    219   15
   Dalton Kincaid       TE    172    9


--- Simulating drafting 'Bucky Irving' for $5 ---


--- Simulating drafting 'Puka Nacua' for $10 ---


--- Simulating drafting 'Ja'Marr Chase' for $64 ---
--- Finding New Optimal Lineup (Remaining 5 Players) ---
Status: Optimal
New Total Projected Points: 2216
New Total Projected Cost: $179

New Optimal Lineup (after 'Bucky Irving', 'Puka Nacua', and 'Ja'Marr Chase' were drafted):
           Player Position Points Cost
       Joe Burrow       QB    361   20
     Bucky Irving       RB    264  

In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: DEFINE A FUNCTION TO SOLVE THE OPTIMIZATION PROBLEM ---
def find_optimal_lineup(eligible_players_df, drafted_players, initial_budget):
    """
    Finds the optimal fantasy football lineup using PuLP Linear Programming.

    This function dynamically adjusts constraints based on the players already drafted.

    Args:
        eligible_players_df (pd.DataFrame): DataFrame of available players.
        drafted_players (pd.DataFrame): DataFrame of players already on your team.
        initial_budget (int): The starting budget for the entire draft.

    Returns:
        A tuple containing:
        - status (str): The status of the solver ('Optimal', 'Infeasible', etc.).
        - total_projected_points (float): The total projected points of the final lineup.
        - total_actual_cost (float): The total cost of the final lineup.
        - final_lineup_df (pd.DataFrame): The complete DataFrame of the selected players.
    """
    # Calculate current state based on drafted players
    players_to_draft = 8 - len(drafted_players)
    budget_spent = drafted_players['Cost'].sum()
    remaining_budget = initial_budget - budget_spent

    # Calculate remaining positional needs
    qbs_needed = 1 - len(drafted_players[drafted_players['Position'] == 'QB'])
    rbs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'RB'])
    wrs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'WR'])
    tes_needed = 1 - len(drafted_players[drafted_players['Position'] == 'TE'])

    # Ensure needs are not negative
    qbs_needed = max(0, qbs_needed)
    rbs_needed = max(0, rbs_needed)
    wrs_needed = max(0, wrs_needed)
    tes_needed = max(0, tes_needed)

    # Calculate total RB/WR/TE spots needed for the new problem
    flex_players_drafted = len(drafted_players[drafted_players['Position'].isin(['RB', 'WR', 'TE'])])
    flex_players_needed = 7 - flex_players_drafted

    # Create the linear programming problem to maximize total points
    prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)

    # A binary variable for each player to decide if they are in the lineup (1) or not (0)
    player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

    # Objective Function: Maximize the sum of points for selected players
    prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

    # Constraint 1: The total cost of the new players cannot exceed the remaining budget
    prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= remaining_budget, "Remaining Budget Constraint"

    # Constraint 2: Positional requirements for a valid lineup
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB') == qbs_needed, "QB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= rbs_needed, "RB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= wrs_needed, "WR Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= tes_needed, "TE Constraint"

    # Constraint 3: The flex positions - a total of (players_to_draft - qbs_needed) RB/WR/TE players are required
    flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
    prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == flex_players_needed, "Total RB/WR/TE"

    # Constraint 4: The total number of players must match the number we are drafting
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == players_to_draft, "Total Roster Size"

    # Solve the optimization problem
    prob.solve()
    status = pulp.LpStatus[prob.status]

    # Extract the newly selected players and create a DataFrame
    newly_selected_players = [
        eligible_players_df.loc[i]
        for i in eligible_players_df.index
        if player_vars[i].varValue == 1
    ]
    newly_selected_df = pd.DataFrame(newly_selected_players)

    # Combine the drafted players with the newly selected optimal players
    final_lineup_df = pd.concat([drafted_players, newly_selected_df], ignore_index=True)

    # Calculate the total points and cost of the final lineup
    total_projected_points = final_lineup_df['Points'].sum()
    total_actual_cost = final_lineup_df['Cost'].sum()

    # Sort the final lineup for a cleaner display
    position_order = ['QB', 'RB', 'WR', 'TE']
    final_lineup_df['Position_Order'] = pd.Categorical(
        final_lineup_df['Position'], categories=position_order, ordered=True
    )
    final_lineup_df = final_lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

    return status, total_projected_points, total_actual_cost, final_lineup_df


# --- STEP 2: LOAD THE CSV DATA AND INITIALIZE ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
initial_budget = 179

# Start with an empty drafted players DataFrame with the correct columns
drafted_players_df = pd.DataFrame(columns=['Player', 'Position', 'Points', 'Cost'])

# --- STEP 3: FIND THE INITIAL OPTIMAL LINEUP (full 8 players) ---
print("--- Finding Initial Optimal Lineup (Full Roster) ---")
status, points, cost, final_lineup_df = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {status}")
print(f"Total Projected Points: {points}")
print(f"Total Projected Cost: ${cost}\n")
print("Initial Optimal Lineup:")
print(final_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 8: CALCULATE BREAKING POINT COSTS FOR THE INITIAL OPTIMAL LINEUP ---
# This new section calculates the maximum a player can cost before they are no longer
# considered a part of the optimal lineup.
print("\n\n--- CALCULATING BREAKING POINT COSTS ---")
breaking_points = {}
# The players to test are the ones in the initial optimal lineup.
players_to_test_df = final_lineup_df.copy()

# Iterate through each player in the newly optimized lineup
for index, player_to_test in players_to_test_df.iterrows():
    player_name = player_to_test['Player']

    # Start the test cost at the player's current projected cost
    test_cost = player_to_test['Cost']

    # Keep increasing the cost by $1 until the player is no longer in the lineup
    while True:
        test_cost += 1

        # Create a temporary eligible player DataFrame for this test, starting with the full pool
        temp_eligible_df = eligible_players_df.copy()

        # Find the player in the temporary DataFrame and update their cost
        temp_eligible_df.loc[temp_eligible_df['Player'] == player_name, 'Cost'] = test_cost

        # Re-run the optimizer with the modified cost for this one player
        # The key change is to run the optimizer on the full pool with only the one player's cost modified.
        temp_status, _, _, temp_final_lineup_df = find_optimal_lineup(
            temp_eligible_df,
            drafted_players_df,
            initial_budget
        )

        # Check if the player is no longer in the optimal lineup
        if player_name not in temp_final_lineup_df['Player'].tolist():
            breaking_points[player_name] = test_cost - 1
            break  # Exit the while loop for this player

        # Safety break to prevent infinite loops if a player is always optimal
        if test_cost > initial_budget:
            breaking_points[player_name] = 'Irreplaceable'
            break

# Print the breaking point costs
if breaking_points:
    print("Breaking point costs for the initial optimal players:")
    for player, cost in breaking_points.items():
        print(f"- {player}: ${cost}")
else:
    print("Could not calculate breaking points. Check the lineup and budget.")


--- Finding Initial Optimal Lineup (Full Roster) ---
Status: Optimal
Total Projected Points: 2031
Total Projected Cost: $179

Initial Optimal Lineup:
           Player Position Points Cost
   Jayden Daniels       QB    382   28
      Chase Brown       RB    265   36
      Breece Hall       RB    264   35
      Rashee Rice       WR    259   20
    Jaylen Waddle       WR    237   20
Tetairoa McMillan       WR    233   16
      Chris Olave       WR    219   15
   Dalton Kincaid       TE    172    9


--- CALCULATING BREAKING POINT COSTS ---
Breaking point costs for the initial optimal players:
- Jayden Daniels: $28
- Chase Brown: $38
- Breece Hall: $37
- Rashee Rice: $28
- Jaylen Waddle: $20
- Tetairoa McMillan: $20
- Chris Olave: $15
- Dalton Kincaid: $11


In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: DEFINE A FUNCTION TO SOLVE THE OPTIMIZATION PROBLEM ---
def find_optimal_lineup(eligible_players_df, drafted_players, initial_budget):
    """
    Finds the optimal fantasy football lineup using PuLP Linear Programming.

    This function dynamically adjusts constraints based on the players already drafted.

    Args:
        eligible_players_df (pd.DataFrame): DataFrame of available players.
        drafted_players (pd.DataFrame): DataFrame of players already on your team.
        initial_budget (int): The starting budget for the entire draft.

    Returns:
        A tuple containing:
        - status (str): The status of the solver ('Optimal', 'Infeasible', etc.).
        - total_projected_points (float): The total projected points of the final lineup.
        - total_actual_cost (float): The total cost of the final lineup.
        - final_lineup_df (pd.DataFrame): The complete DataFrame of the selected players.
    """
    # Calculate current state based on drafted players
    players_to_draft = 8 - len(drafted_players)
    budget_spent = drafted_players['Cost'].sum()
    remaining_budget = initial_budget - budget_spent

    # Calculate remaining positional needs
    qbs_needed = 1 - len(drafted_players[drafted_players['Position'] == 'QB'])
    rbs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'RB'])
    wrs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'WR'])
    tes_needed = 1 - len(drafted_players[drafted_players['Position'] == 'TE'])

    # Ensure needs are not negative
    qbs_needed = max(0, qbs_needed)
    rbs_needed = max(0, rbs_needed)
    wrs_needed = max(0, wrs_needed)
    tes_needed = max(0, tes_needed)

    # Calculate total RB/WR/TE spots needed for the new problem
    flex_players_drafted = len(drafted_players[drafted_players['Position'].isin(['RB', 'WR', 'TE'])])
    flex_players_needed = 7 - flex_players_drafted

    # Create the linear programming problem to maximize total points
    prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)

    # A binary variable for each player to decide if they are in the lineup (1) or not (0)
    player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

    # Objective Function: Maximize the sum of points for selected players
    prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

    # Constraint 1: The total cost of the new players cannot exceed the remaining budget
    prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= remaining_budget, "Remaining Budget Constraint"

    # Constraint 2: Positional requirements for a valid lineup
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB') == qbs_needed, "QB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= rbs_needed, "RB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= wrs_needed, "WR Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= tes_needed, "TE Constraint"

    # Constraint 3: The flex positions - a total of (players_to_draft - qbs_needed) RB/WR/TE players are required
    flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
    prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == flex_players_needed, "Total RB/WR/TE"

    # Constraint 4: The total number of players must match the number we are drafting
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == players_to_draft, "Total Roster Size"

    # Solve the optimization problem
    prob.solve()
    status = pulp.LpStatus[prob.status]

    # Extract the newly selected players and create a DataFrame
    newly_selected_players = [
        eligible_players_df.loc[i]
        for i in eligible_players_df.index
        if player_vars[i].varValue == 1
    ]
    newly_selected_df = pd.DataFrame(newly_selected_players)

    # Combine the drafted players with the newly selected optimal players
    final_lineup_df = pd.concat([drafted_players, newly_selected_df], ignore_index=True)

    # Calculate the total points and cost of the final lineup
    total_projected_points = final_lineup_df['Points'].sum()
    total_actual_cost = final_lineup_df['Cost'].sum()

    # Sort the final lineup for a cleaner display
    position_order = ['QB', 'RB', 'WR', 'TE']
    final_lineup_df['Position_Order'] = pd.Categorical(
        final_lineup_df['Position'], categories=position_order, ordered=True
    )
    final_lineup_df = final_lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

    return status, total_projected_points, total_actual_cost, final_lineup_df


# --- STEP 2: LOAD THE CSV DATA AND INITIALIZE ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
initial_budget = 179

# Start with an empty drafted players DataFrame with the correct columns
drafted_players_df = pd.DataFrame(columns=['Player', 'Position', 'Points', 'Cost'])

# --- NEW: STEP 2A: SIMULATE DRAFTED PLAYERS ---
# We will remove these players from the eligible pool to simulate a draft in progress
drafted_by_others = [
    'Jalen Hurts',
    'Alvin Kamara',
    'Lamar Jackson',
    'Rashee Rice',
    'Dalton Kincaid'
]

print("--- Simulating Draft in Progress ---")
print(f"The following players have been drafted by other teams and are no longer available: {', '.join(drafted_by_others)}\n")

# Filter the main DataFrame to remove the simulated drafted players
eligible_players_df = eligible_players_df[~eligible_players_df['Player'].isin(drafted_by_others)].reset_index(drop=True)


# --- STEP 3: FIND THE INITIAL OPTIMAL LINEUP (full 8 players) ---
print("--- Finding Initial Optimal Lineup (Full Roster) ---")
status, points, cost, final_lineup_df = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {status}")
print(f"Total Projected Points: {points}")
print(f"Total Projected Cost: ${cost}\n")
print("Initial Optimal Lineup:")
print(final_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 8: CALCULATE BREAKING POINT COSTS FOR THE INITIAL OPTIMAL LINEUP ---
# This new section calculates the maximum a player can cost before they are no longer
# considered a part of the optimal lineup.
print("\n\n--- CALCULATING BREAKING POINT COSTS ---")
breaking_points = {}
# The players to test are the ones in the initial optimal lineup.
players_to_test_df = final_lineup_df.copy()

# Iterate through each player in the newly optimized lineup
for index, player_to_test in players_to_test_df.iterrows():
    player_name = player_to_test['Player']

    # Start the test cost at the player's current projected cost
    test_cost = player_to_test['Cost']

    # Keep increasing the cost by $1 until the player is no longer in the lineup
    while True:
        test_cost += 1

        # Create a temporary eligible player DataFrame for this test, starting with the full pool
        temp_eligible_df = eligible_players_df.copy()

        # Find the player in the temporary DataFrame and update their cost
        temp_eligible_df.loc[temp_eligible_df['Player'] == player_name, 'Cost'] = test_cost

        # Re-run the optimizer with the modified cost for this one player
        # The key change is to run the optimizer on the full pool with only the one player's cost modified.
        temp_status, _, _, temp_final_lineup_df = find_optimal_lineup(
            temp_eligible_df,
            drafted_players_df,
            initial_budget
        )

        # Check if the player is no longer in the optimal lineup
        if player_name not in temp_final_lineup_df['Player'].tolist():
            breaking_points[player_name] = test_cost - 1
            break  # Exit the while loop for this player

        # Safety break to prevent infinite loops if a player is always optimal
        if test_cost > initial_budget:
            breaking_points[player_name] = 'Irreplaceable'
            break

# Print the breaking point costs
if breaking_points:
    print("Breaking point costs for the initial optimal players:")
    for player, cost in breaking_points.items():
        print(f"- {player}: ${cost}")
else:
    print("Could not calculate breaking points. Check the lineup and budget.")


--- Simulating Draft in Progress ---
The following players have been drafted by other teams and are no longer available: Jalen Hurts, Alvin Kamara, Lamar Jackson, Rashee Rice, Dalton Kincaid

--- Finding Initial Optimal Lineup (Full Roster) ---
Status: Optimal
Total Projected Points: 2002
Total Projected Cost: $179

Initial Optimal Lineup:
           Player Position Points Cost
   Jayden Daniels       QB    382   28
      Chase Brown       RB    265   36
      Breece Hall       RB    264   35
    Jaylen Waddle       WR    237   20
Tetairoa McMillan       WR    233   16
     Chris Godwin       WR    221   17
    Khalil Shakir       WR    211   11
      David Njoku       TE    189   16


--- CALCULATING BREAKING POINT COSTS ---
Breaking point costs for the initial optimal players:
- Jayden Daniels: $28
- Chase Brown: $38
- Breece Hall: $37
- Jaylen Waddle: $22
- Tetairoa McMillan: $19
- Chris Godwin: $17
- Khalil Shakir: $13
- David Njoku: $17


In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: DEFINE A FUNCTION TO SOLVE THE OPTIMIZATION PROBLEM ---
def find_optimal_lineup(eligible_players_df, drafted_players, initial_budget):
    """
    Finds the optimal fantasy football lineup using PuLP Linear Programming.

    This function dynamically adjusts constraints based on the players already drafted.

    Args:
        eligible_players_df (pd.DataFrame): DataFrame of available players.
        drafted_players (pd.DataFrame): DataFrame of players already on your team.
        initial_budget (int): The starting budget for the entire draft.

    Returns:
        A tuple containing:
        - status (str): The status of the solver ('Optimal', 'Infeasible', etc.).
        - total_projected_points (float): The total projected points of the final lineup.
        - total_actual_cost (float): The total cost of the final lineup.
        - final_lineup_df (pd.DataFrame): The complete DataFrame of the selected players.
    """
    # Calculate current state based on drafted players
    players_to_draft = 8 - len(drafted_players)
    budget_spent = drafted_players['Cost'].sum()
    remaining_budget = initial_budget - budget_spent

    # Calculate remaining positional needs
    qbs_needed = 1 - len(drafted_players[drafted_players['Position'] == 'QB'])
    rbs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'RB'])
    wrs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'WR'])
    tes_needed = 1 - len(drafted_players[drafted_players['Position'] == 'TE'])

    # Ensure needs are not negative
    qbs_needed = max(0, qbs_needed)
    rbs_needed = max(0, rbs_needed)
    wrs_needed = max(0, wrs_needed)
    tes_needed = max(0, tes_needed)

    # Calculate total RB/WR/TE spots needed for the new problem
    flex_players_drafted = len(drafted_players[drafted_players['Position'].isin(['RB', 'WR', 'TE'])])
    flex_players_needed = 7 - flex_players_drafted

    # Create the linear programming problem to maximize total points
    prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)

    # A binary variable for each player to decide if they are in the lineup (1) or not (0)
    player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

    # Objective Function: Maximize the sum of points for selected players
    prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

    # Constraint 1: The total cost of the new players cannot exceed the remaining budget
    prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= remaining_budget, "Remaining Budget Constraint"

    # Constraint 2: Positional requirements for a valid lineup
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB') == qbs_needed, "QB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= rbs_needed, "RB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= wrs_needed, "WR Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= tes_needed, "TE Constraint"

    # Constraint 3: The flex positions - a total of (players_to_draft - qbs_needed) RB/WR/TE players are required
    flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
    prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == flex_players_needed, "Total RB/WR/TE"

    # Constraint 4: The total number of players must match the number we are drafting
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == players_to_draft, "Total Roster Size"

    # Solve the optimization problem
    prob.solve()
    status = pulp.LpStatus[prob.status]

    # Extract the newly selected players and create a DataFrame
    newly_selected_players = [
        eligible_players_df.loc[i]
        for i in eligible_players_df.index
        if player_vars[i].varValue == 1
    ]
    newly_selected_df = pd.DataFrame(newly_selected_players)

    # Combine the drafted players with the newly selected optimal players
    final_lineup_df = pd.concat([drafted_players, newly_selected_df], ignore_index=True)

    # Calculate the total points and cost of the final lineup
    total_projected_points = final_lineup_df['Points'].sum()
    total_actual_cost = final_lineup_df['Cost'].sum()

    # Sort the final lineup for a cleaner display
    position_order = ['QB', 'RB', 'WR', 'TE']
    final_lineup_df['Position_Order'] = pd.Categorical(
        final_lineup_df['Position'], categories=position_order, ordered=True
    )
    final_lineup_df = final_lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

    return status, total_projected_points, total_actual_cost, final_lineup_df


# --- STEP 2: LOAD THE CSV DATA AND INITIALIZE ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
initial_budget = 179

# Start with an empty drafted players DataFrame with the correct columns
drafted_players_df = pd.DataFrame(columns=['Player', 'Position', 'Points', 'Cost'])

# --- SIMULATION REMOVED: NO PLAYERS ARE DRAFTED YET ---
print("--- Starting with a clean draft board ---")
# The eligible_players_df is not filtered and remains the full list.


# --- STEP 3: FIND THE INITIAL OPTIMAL LINEUP (full 8 players) ---
print("--- Finding Initial Optimal Lineup (Full Roster) ---")
status, points, cost, final_lineup_df = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {status}")
print(f"Total Projected Points: {points}")
print(f"Total Projected Cost: ${cost}\n")
print("Initial Optimal Lineup:")
print(final_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 8: CALCULATE BREAKING POINT COSTS FOR THE INITIAL OPTIMAL LINEUP ---
# This new section calculates the maximum a player can cost before they are no longer
# considered a part of the optimal lineup.
print("\n\n--- CALCULATING BREAKING POINT COSTS ---")
breaking_points = {}
# The players to test are the ones in the initial optimal lineup.
players_to_test_df = final_lineup_df.copy()

# Iterate through each player in the newly optimized lineup
for index, player_to_test in players_to_test_df.iterrows():
    player_name = player_to_test['Player']

    # Start the test cost at the player's current projected cost
    test_cost = player_to_test['Cost']

    # Keep increasing the cost by $1 until the player is no longer in the lineup
    while True:
        test_cost += 1

        # Create a temporary eligible player DataFrame for this test, starting with the full pool
        temp_eligible_df = eligible_players_df.copy()

        # Find the player in the temporary DataFrame and update their cost
        temp_eligible_df.loc[temp_eligible_df['Player'] == player_name, 'Cost'] = test_cost

        # Re-run the optimizer with the modified cost for this one player
        # The key change is to run the optimizer on the full pool with only the one player's cost modified.
        temp_status, _, _, temp_final_lineup_df = find_optimal_lineup(
            temp_eligible_df,
            drafted_players_df,
            initial_budget
        )

        # Check if the player is no longer in the optimal lineup
        if player_name not in temp_final_lineup_df['Player'].tolist():
            breaking_points[player_name] = test_cost - 1
            break  # Exit the while loop for this player

        # Safety break to prevent infinite loops if a player is always optimal
        if test_cost > initial_budget:
            breaking_points[player_name] = 'Irreplaceable'
            break

# Print the breaking point costs
if breaking_points:
    print("Breaking point costs for the initial optimal players:")
    for player, cost in breaking_points.items():
        print(f"- {player}: ${cost}")
else:
    print("Could not calculate breaking points. Check the lineup and budget.")


# --- NEW: STEP 9: CALCULATE "GOOD VALUE" COSTS FOR NON-OPTIMAL PLAYERS ---
# This new section calculates the cost at which players not in the optimal lineup
# would become part of the optimal lineup. This is the "good value" price.
print("\n\n--- CALCULATING 'GOOD VALUE' COSTS FOR NON-OPTIMAL PLAYERS ---")
good_value_costs = {}

# Get the list of players currently in the optimal lineup
optimal_player_names = final_lineup_df['Player'].tolist()
# Get the list of players not in the optimal lineup
non_optimal_players_df = eligible_players_df[~eligible_players_df['Player'].isin(optimal_player_names)].reset_index(drop=True)

# Iterate through each non-optimal player
for index, player_to_test in non_optimal_players_df.iterrows():
    player_name = player_to_test['Player']
    original_cost = player_to_test['Cost']

    # Start the test cost at the player's projected cost
    test_cost = original_cost

    # Keep decreasing the cost by $1 until the player is included in the optimal lineup
    while test_cost >= 1:
        # Create a temporary eligible player DataFrame, starting with the full pool
        temp_eligible_df = eligible_players_df.copy()

        # Find the player in the temporary DataFrame and update their cost
        temp_eligible_df.loc[temp_eligible_df['Player'] == player_name, 'Cost'] = test_cost

        # Re-run the optimizer with the modified cost for this one player
        temp_status, _, _, temp_final_lineup_df = find_optimal_lineup(
            temp_eligible_df,
            drafted_players_df,
            initial_budget
        )

        # Check if the player is now in the optimal lineup
        if player_name in temp_final_lineup_df['Player'].tolist():
            good_value_costs[player_name] = test_cost
            break # Exit the while loop for this player

        # Decrease the test cost
        test_cost -= 1

# Print the good value costs
if good_value_costs:
    print("Good value costs for other available players:")
    for player, cost in good_value_costs.items():
        # Only print players who actually have a good value cost (i.e. became part of the optimal lineup)
        # and whose good value is less than their original projected cost.
        if cost < eligible_players_df[eligible_players_df['Player'] == player]['Cost'].iloc[0]:
            print(f"- {player}: ${cost} (Projected: ${eligible_players_df[eligible_players_df['Player'] == player]['Cost'].iloc[0]})")
else:
    print("No other players could enter the optimal lineup by reducing their cost.")

--- Starting with a clean draft board ---
--- Finding Initial Optimal Lineup (Full Roster) ---
Status: Optimal
Total Projected Points: 2031
Total Projected Cost: $179

Initial Optimal Lineup:
           Player Position Points Cost
   Jayden Daniels       QB    382   28
      Chase Brown       RB    265   36
      Breece Hall       RB    264   35
      Rashee Rice       WR    259   20
    Jaylen Waddle       WR    237   20
Tetairoa McMillan       WR    233   16
      Chris Olave       WR    219   15
   Dalton Kincaid       TE    172    9


--- CALCULATING BREAKING POINT COSTS ---
Breaking point costs for the initial optimal players:
- Jayden Daniels: $28
- Chase Brown: $38
- Breece Hall: $37
- Rashee Rice: $28
- Jaylen Waddle: $20
- Tetairoa McMillan: $20
- Chris Olave: $15
- Dalton Kincaid: $11


--- CALCULATING 'GOOD VALUE' COSTS FOR NON-OPTIMAL PLAYERS ---
Good value costs for other available players:
- Lamar Jackson: $28 (Projected: $29)
- Josh Allen: $26 (Projected: $29)
- Jalen Hu

In [ ]:
# --- This line is for Google Colab/Jupyter to install the required library ---
# If you are running this in a local Python environment, you can remove the "!"
!pip install pulp

import pandas as pd
import pulp

# --- STEP 1: DEFINE A FUNCTION TO SOLVE THE OPTIMIZATION PROBLEM ---
def find_optimal_lineup(eligible_players_df, drafted_players, initial_budget):
    """
    Finds the optimal fantasy football lineup using PuLP Linear Programming.

    This function dynamically adjusts constraints based on the players already drafted.

    Args:
        eligible_players_df (pd.DataFrame): DataFrame of available players.
        drafted_players (pd.DataFrame): DataFrame of players already on your team.
        initial_budget (int): The starting budget for the entire draft.

    Returns:
        A tuple containing:
        - status (str): The status of the solver ('Optimal', 'Infeasible', etc.).
        - total_projected_points (float): The total projected points of the final lineup.
        - total_actual_cost (float): The total cost of the final lineup.
        - final_lineup_df (pd.DataFrame): The complete DataFrame of the selected players.
    """
    # Calculate current state based on drafted players
    players_to_draft = 8 - len(drafted_players)
    budget_spent = drafted_players['Cost'].sum()
    remaining_budget = initial_budget - budget_spent

    # Calculate remaining positional needs
    qbs_needed = 1 - len(drafted_players[drafted_players['Position'] == 'QB'])
    rbs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'RB'])
    wrs_needed = 2 - len(drafted_players[drafted_players['Position'] == 'WR'])
    tes_needed = 1 - len(drafted_players[drafted_players['Position'] == 'TE'])

    # Ensure needs are not negative
    qbs_needed = max(0, qbs_needed)
    rbs_needed = max(0, rbs_needed)
    wrs_needed = max(0, wrs_needed)
    tes_needed = max(0, tes_needed)

    # Calculate total RB/WR/TE spots needed for the new problem
    flex_players_drafted = len(drafted_players[drafted_players['Position'].isin(['RB', 'WR', 'TE'])])
    flex_players_needed = 7 - flex_players_drafted

    # Create the linear programming problem to maximize total points
    prob = pulp.LpProblem("Fantasy_Football_Lineup", pulp.LpMaximize)

    # A binary variable for each player to decide if they are in the lineup (1) or not (0)
    player_vars = pulp.LpVariable.dicts("player", eligible_players_df.index, 0, 1, pulp.LpBinary)

    # Objective Function: Maximize the sum of points for selected players
    prob += pulp.lpSum(eligible_players_df['Points'][i] * player_vars[i] for i in eligible_players_df.index), "Total Projected Points"

    # Constraint 1: The total cost of the new players cannot exceed the remaining budget
    prob += pulp.lpSum(eligible_players_df['Cost'][i] * player_vars[i] for i in eligible_players_df.index) <= remaining_budget, "Remaining Budget Constraint"

    # Constraint 2: Positional requirements for a valid lineup
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'QB') == qbs_needed, "QB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'RB') >= rbs_needed, "RB Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'WR') >= wrs_needed, "WR Constraint"
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index if eligible_players_df['Position'][i] == 'TE') >= tes_needed, "TE Constraint"

    # Constraint 3: The flex positions - a total of (players_to_draft - qbs_needed) RB/WR/TE players are required
    flex_eligible_players_indexes = eligible_players_df.index[eligible_players_df['Position'].isin(['RB', 'WR', 'TE'])]
    prob += pulp.lpSum(player_vars[i] for i in flex_eligible_players_indexes) == flex_players_needed, "Total RB/WR/TE"

    # Constraint 4: The total number of players must match the number we are drafting
    prob += pulp.lpSum(player_vars[i] for i in eligible_players_df.index) == players_to_draft, "Total Roster Size"

    # Solve the optimization problem
    prob.solve()
    status = pulp.LpStatus[prob.status]

    # Extract the newly selected players and create a DataFrame
    newly_selected_players = [
        eligible_players_df.loc[i]
        for i in eligible_players_df.index
        if player_vars[i].varValue == 1
    ]
    newly_selected_df = pd.DataFrame(newly_selected_players)

    # Combine the drafted players with the newly selected optimal players
    final_lineup_df = pd.concat([drafted_players, newly_selected_df], ignore_index=True)

    # Calculate the total points and cost of the final lineup
    total_projected_points = final_lineup_df['Points'].sum()
    total_actual_cost = final_lineup_df['Cost'].sum()

    # Sort the final lineup for a cleaner display
    position_order = ['QB', 'RB', 'WR', 'TE']
    final_lineup_df['Position_Order'] = pd.Categorical(
        final_lineup_df['Position'], categories=position_order, ordered=True
    )
    final_lineup_df = final_lineup_df.sort_values(by=['Position_Order', 'Points'], ascending=[True, False])

    return status, total_projected_points, total_actual_cost, final_lineup_df


# --- STEP 2: LOAD THE CSV DATA AND INITIALIZE ---
# Ensure you have uploaded your CSV file to the Colab session
file_path = '/content/python csv test file 7-31.csv'
df = pd.read_csv(file_path)

# Clean and filter the data to only include eligible starting positions
df['Position'] = df['Position'].str.strip().str.upper()
eligible_players_df = df[df['Position'].isin(['QB', 'RB', 'WR', 'TE'])].reset_index(drop=True)

# Define the budget for the lineup
initial_budget = 179

# --- STEP 2A: SIMULATE DRAFTED PLAYERS ---
# The players you have drafted
drafted_by_me = [
    {'Player': 'Bucky Irving', 'Position': 'RB', 'Points': 105.8, 'Cost': 5},
    {'Player': 'Puka Nacua', 'Position': 'WR', 'Points': 193.3, 'Cost': 10}
]
drafted_players_df = pd.DataFrame(drafted_by_me)

# The players drafted by others
drafted_by_others = [
    'Lamar Jackson',
    'Josh Allen',
    'Jayden Daniels',
    'Jalen Hurts',
    'Chase Brown',
    'Rashee Rice'
]

print("--- Simulating Draft in Progress ---")
print(f"You have drafted: {', '.join(p['Player'] for p in drafted_by_me)}\n")
print(f"The following players have been drafted by other teams: {', '.join(drafted_by_others)}\n")

# Filter the main DataFrame to remove the simulated drafted players
eligible_players_df = eligible_players_df[
    ~eligible_players_df['Player'].isin(drafted_by_others + [p['Player'] for p in drafted_by_me])
].reset_index(drop=True)

# --- STEP 3: FIND THE OPTIMAL LINEUP FOR THE REMAINING SPOTS ---
print("--- Finding New Optimal Lineup ---")
status, points, cost, final_lineup_df = find_optimal_lineup(eligible_players_df, drafted_players_df, initial_budget)

print(f"Status: {status}")
print(f"Total Projected Points: {points}")
print(f"Total Projected Cost: ${cost}\n")
print("New Optimal Lineup:")
print(final_lineup_df[['Player', 'Position', 'Points', 'Cost']].to_string(index=False))

# --- STEP 8: CALCULATE BREAKING POINT COSTS FOR THE NEW OPTIMAL LINEUP ---
# This new section calculates the maximum a player can cost before they are no longer
# considered a part of the optimal lineup.
print("\n\n--- CALCULATING BREAKING POINT COSTS ---")
breaking_points = {}
# The players to test are the ones in the new optimal lineup.
players_to_test_df = final_lineup_df.copy()

# Iterate through each player in the newly optimized lineup
for index, player_to_test in players_to_test_df.iterrows():
    player_name = player_to_test['Player']

    # Start the test cost at the player's current projected cost
    test_cost = player_to_test['Cost']

    # Keep increasing the cost by $1 until the player is no longer in the lineup
    while True:
        test_cost += 1

        # Create a temporary eligible player DataFrame for this test, starting with the full pool
        temp_eligible_df = eligible_players_df.copy()

        # Find the player in the temporary DataFrame and update their cost
        temp_eligible_df.loc[temp_eligible_df['Player'] == player_name, 'Cost'] = test_cost

        # Re-run the optimizer with the modified cost for this one player
        # The key change is to run the optimizer on the full pool with only the one player's cost modified.
        temp_status, _, _, temp_final_lineup_df = find_optimal_lineup(
            temp_eligible_df,
            drafted_players_df,
            initial_budget
        )

        # Check if the player is no longer in the optimal lineup
        if player_name not in temp_final_lineup_df['Player'].tolist():
            breaking_points[player_name] = test_cost - 1
            break  # Exit the while loop for this player

        # Safety break to prevent infinite loops if a player is always optimal
        if test_cost > initial_budget:
            breaking_points[player_name] = 'Irreplaceable'
            break

# Print the breaking point costs
if breaking_points:
    print("Breaking point costs for the new optimal players:")
    for player, cost in breaking_points.items():
        print(f"- {player}: ${cost}")
else:
    print("Could not calculate breaking points. Check the lineup and budget.")


# --- NEW: STEP 9: CALCULATE "GOOD VALUE" COSTS FOR NON-OPTIMAL PLAYERS ---
# This new section calculates the cost at which players not in the optimal lineup
# would become part of the optimal lineup. This is the "good value" price.
print("\n\n--- CALCULATING 'GOOD VALUE' COSTS FOR NON-OPTIMAL PLAYERS ---")
good_value_costs = {}

# Get the list of players currently in the optimal lineup
optimal_player_names = final_lineup_df['Player'].tolist()
# Get the list of players not in the optimal lineup
non_optimal_players_df = eligible_players_df[~eligible_players_df['Player'].isin(optimal_player_names)].reset_index(drop=True)

# Iterate through each non-optimal player
for index, player_to_test in non_optimal_players_df.iterrows():
    player_name = player_to_test['Player']
    original_cost = player_to_test['Cost']

    # Start the test cost at the player's projected cost
    test_cost = original_cost

    # Keep decreasing the cost by $1 until the player is included in the optimal lineup
    while test_cost >= 1:
        # Create a temporary eligible player DataFrame, starting with the full pool
        temp_eligible_df = eligible_players_df.copy()

        # Find the player in the temporary DataFrame and update their cost
        temp_eligible_df.loc[temp_eligible_df['Player'] == player_name, 'Cost'] = test_cost

        # Re-run the optimizer with the modified cost for this one player
        temp_status, _, _, temp_final_lineup_df = find_optimal_lineup(
            temp_eligible_df,
            drafted_players_df,
            initial_budget
        )

        # Check if the player is now in the optimal lineup
        if player_name in temp_final_lineup_df['Player'].tolist():
            good_value_costs[player_name] = test_cost
            break # Exit the while loop for this player

        # Decrease the test cost
        test_cost -= 1

# Print the good value costs
if good_value_costs:
    print("Good value costs for other available players:")
    for player, cost in good_value_costs.items():
        # Only print players who actually have a good value cost (i.e. became part of the optimal lineup)
        # and whose good value is less than their original projected cost.
        if cost < eligible_players_df[eligible_players_df['Player'] == player]['Cost'].iloc[0]:
            print(f"- {player}: ${cost} (Projected: ${eligible_players_df[eligible_players_df['Player'] == player]['Cost'].iloc[0]})")
else:
    print("No other players could enter the optimal lineup by reducing their cost.")


--- Simulating Draft in Progress ---
You have drafted: Bucky Irving, Puka Nacua

The following players have been drafted by other teams: Lamar Jackson, Josh Allen, Jayden Daniels, Jalen Hurts, Chase Brown, Rashee Rice

--- Finding New Optimal Lineup ---
Status: Optimal
Total Projected Points: 1902.1
Total Projected Cost: $178

New Optimal Lineup:
           Player Position  Points  Cost
       Joe Burrow       QB   361.0    20
      Breece Hall       RB   264.0    35
     Bucky Irving       RB   105.8     5
       D.J. Moore       WR   247.0    26
    Jaylen Waddle       WR   237.0    20
Tetairoa McMillan       WR   233.0    16
       Puka Nacua       WR   193.3    10
     Brock Bowers       TE   261.0    46


--- CALCULATING BREAKING POINT COSTS ---
Breaking point costs for the new optimal players:
- Joe Burrow: $24
- Breece Hall: $39
- Bucky Irving: $Irreplaceable
- D.J. Moore: $30
- Jaylen Waddle: $24
- Tetairoa McMillan: $21
- Puka Nacua: $Irreplaceable
- Brock Bowers: $47


--- CA